<a href="https://colab.research.google.com/github/niofrequency/google-collab_image/blob/main/QWEN-AIO-NSFW-EDIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: install ComfyUI + ComfyUI-GGUF and download the Qwen Rapid AIO edit models
%cd /content
!git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI || true
!cd /content/ComfyUI && git pull -q || true
!cd /content/ComfyUI && pip install -q -r requirements.txt
!pip install -q torchsde gguf fastapi uvicorn pydantic requests nest_asyncio

# ComfyUI-GGUF for loading .gguf diffusion models
!cd /content/ComfyUI/custom_nodes && (test -d ComfyUI-GGUF || git clone -q https://github.com/city96/ComfyUI-GGUF.git)
!cd /content/ComfyUI/custom_nodes/ComfyUI-GGUF && pip install -q -r requirements.txt

!apt install -qqy aria2

!mkdir -p /content/ComfyUI/models/unet
!mkdir -p /content/ComfyUI/models/diffusion_models
!mkdir -p /content/ComfyUI/models/text_encoders
!mkdir -p /content/ComfyUI/models/vae

# 1. Qwen-Rapid-NSFW-v23 Q2_K GGUF (7.44 GB)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Novice25/Qwen-Image-Edit-Rapid-AIO-GGUF/resolve/main/v23/Qwen-Rapid-NSFW-v23_Q2_K.gguf \
  -d /content/ComfyUI/models/unet -o Qwen-Rapid-NSFW-v23_Q2_K.gguf

# 2. Qwen 2.5 VL 7B FP8 text encoder
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors \
  -d /content/ComfyUI/models/text_encoders -o qwen_2.5_vl_7b_fp8_scaled.safetensors

# 3. Qwen Image VAE
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M \
  https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors \
  -d /content/ComfyUI/models/vae -o qwen_image_vae.safetensors

!ls -lh /content/ComfyUI/models/unet/
!ls -lh /content/ComfyUI/models/text_encoders/
!ls -lh /content/ComfyUI/models/vae/

In [ ]:
# Cell 2: load the models and define the Qwen image-edit helpers used by cells 3 and 4
%cd /content/ComfyUI

import os
import sys
import gc
import math
import logging
import warnings
import importlib.util

import torch
import numpy as np
from PIL import Image

logging.getLogger("root").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

if "/content/ComfyUI" not in sys.path:
    sys.path.insert(0, "/content/ComfyUI")

import folder_paths
import nodes
import node_helpers
import comfy.utils

MODEL_NAME = "Qwen-Rapid-NSFW-v23_Q2_K.gguf"
CLIP_NAME = "qwen_2.5_vl_7b_fp8_scaled.safetensors"
VAE_NAME = "qwen_image_vae.safetensors"

# Recommended settings for Rapid AIO v23 (distilled: few steps, CFG 1)
AIO_DEFAULTS = {
    "steps": 4,
    "cfg": 1.0,
    "sampler": "sa_solver",
    "scheduler": "beta",
    "width": 1024,
    "height": 1024,
    "denoise": 1.0,
}

# --- Load ComfyUI-GGUF (custom nodes are not auto-loaded when ComfyUI is used as a library)
if "UnetLoaderGGUF" not in nodes.NODE_CLASS_MAPPINGS:
    gguf_path = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF/__init__.py"
    spec = importlib.util.spec_from_file_location(
        "ComfyUI-GGUF", gguf_path,
        submodule_search_locations=[os.path.dirname(gguf_path)],
    )
    module = importlib.util.module_from_spec(spec)
    sys.modules["ComfyUI-GGUF"] = module
    spec.loader.exec_module(module)
    nodes.NODE_CLASS_MAPPINGS.update(getattr(module, "NODE_CLASS_MAPPINGS", {}))
    nodes.NODE_DISPLAY_NAME_MAPPINGS.update(getattr(module, "NODE_DISPLAY_NAME_MAPPINGS", {}))

UnetLoaderGGUF = nodes.NODE_CLASS_MAPPINGS["UnetLoaderGGUF"]()
CLIPLoader = nodes.NODE_CLASS_MAPPINGS["CLIPLoader"]()
VAELoader = nodes.NODE_CLASS_MAPPINGS["VAELoader"]()
KSampler = nodes.NODE_CLASS_MAPPINGS["KSampler"]()
VAEEncode = nodes.NODE_CLASS_MAPPINGS["VAEEncode"]()
VAEDecode = nodes.NODE_CLASS_MAPPINGS["VAEDecode"]()
ConditioningZeroOut = nodes.NODE_CLASS_MAPPINGS["ConditioningZeroOut"]()

gc.collect()
torch.cuda.empty_cache()

with torch.inference_mode():
    unet = UnetLoaderGGUF.load_unet(MODEL_NAME)[0]
    clip = CLIPLoader.load_clip(CLIP_NAME, "qwen_image")[0]
    vae = VAELoader.load_vae(VAE_NAME)[0]


# --- Qwen Image Edit Plus text encoding.
# TextEncodeQwenImageEditPlus lives in comfy_extras and is NOT in nodes.NODE_CLASS_MAPPINGS
# unless ComfyUI's server initialises the extra nodes, so we reproduce it here with the same
# logic: the image goes to the Qwen2.5-VL vision tower (~384x384) AND is VAE-encoded (~1MP)
# as a reference latent. Without both, the model ignores the input image.
QWEN_EDIT_TEMPLATE = (
    "<|im_start|>system\nDescribe the key features of the input image (color, shape, size, texture, "
    "objects, background), then explain how the user's text instruction should alter or modify the "
    "image. Generate a new image that meets the user's requirements while maintaining consistency "
    "with the original input where appropriate.<|im_end|>\n<|im_start|>user\n{}<|im_end|>\n"
    "<|im_start|>assistant\n"
)


def _resize_to_area(samples, total, multiple=1):
    scale = math.sqrt(total / (samples.shape[3] * samples.shape[2]))
    w = max(multiple, round(samples.shape[3] * scale / multiple) * multiple)
    h = max(multiple, round(samples.shape[2] * scale / multiple) * multiple)
    return comfy.utils.common_upscale(samples, w, h, "area", "disabled")


def qwen_edit_encode(clip, prompt, vae=None, images=()):
    """Equivalent of ComfyUI's TextEncodeQwenImageEditPlus. images: list of [1,H,W,3] tensors."""
    images = [im for im in images if im is not None][:3]
    images_vl, ref_latents, image_prompt = [], [], ""
    for i, image in enumerate(images):
        samples = image[:, :, :, :3].movedim(-1, 1)
        images_vl.append(_resize_to_area(samples, 384 * 384).movedim(1, -1))
        if vae is not None:
            ref = _resize_to_area(samples, 1024 * 1024, 8).movedim(1, -1)
            ref_latents.append(vae.encode(ref))
        image_prompt += "Picture {}: <|vision_start|><|image_pad|><|vision_end|>".format(i + 1)

    if images_vl:
        tokens = clip.tokenize(image_prompt + prompt, images=images_vl, llama_template=QWEN_EDIT_TEMPLATE)
    else:
        tokens = clip.tokenize(prompt)
    cond = clip.encode_from_tokens_scheduled(tokens)
    if ref_latents:
        cond = node_helpers.conditioning_set_values(cond, {"reference_latents": ref_latents}, append=True)
    return cond


def pil_to_comfy(pil_img):
    arr = np.array(pil_img.convert("RGB")).astype(np.float32) / 255.0
    return torch.from_numpy(arr)[None, ...]


def comfy_to_pil(comfy_img, index=0):
    arr = (comfy_img[index].detach().cpu().numpy().clip(0.0, 1.0) * 255.0).round().astype(np.uint8)
    return Image.fromarray(arr)


def fit_size(src_w, src_h, req_w, req_h, max_side=1024, min_side=256, multiple=16):
    """Keep the source aspect ratio, aim for about req_w*req_h pixels, clamp and round to /16."""
    area = max(req_w * req_h, min_side * min_side)
    scale = math.sqrt(area / (src_w * src_h))
    w, h = src_w * scale, src_h * scale
    shrink = min(1.0, max_side / max(w, h))
    w, h = w * shrink, h * shrink
    w = int(min(max(round(w / multiple) * multiple, min_side), max_side))
    h = int(min(max(round(h / multiple) * multiple, min_side), max_side))
    return w, h


def run_qwen_edit(positive, negative="", input_pil=None, width=1024, height=1024, steps=4, cfg=1.0,
                  sampler="sa_solver", scheduler="beta", seed=None, denoise=1.0, batch_size=1,
                  on_image=None):
    """Edit input_pil (or generate from scratch when None). Returns (list of PIL images, seeds)."""
    import random
    if input_pil is not None:
        w, h = fit_size(input_pil.width, input_pil.height, width, height)
        input_pil = input_pil.convert("RGB").resize((w, h), Image.Resampling.LANCZOS)
        input_tensor = pil_to_comfy(input_pil)
        images = [input_tensor]
    else:
        w, h = fit_size(width, height, width, height)
        input_tensor = None
        images = []

    seed = seed if seed and seed > 0 else random.randint(1, 2**32 - 1)
    outputs, seeds = [], []
    with torch.inference_mode():
        pos = qwen_edit_encode(clip, positive, vae=vae, images=images)
        if cfg == 1.0:
            # negative is ignored at CFG 1, skip the (slow) VL encode
            neg = ConditioningZeroOut.zero_out(pos)[0]
        else:
            neg = qwen_edit_encode(clip, negative, vae=vae, images=images)

        if input_tensor is not None:
            latent = VAEEncode.encode(vae, input_tensor)[0]
        else:
            latent = {"samples": torch.zeros([1, 16, h // 8, w // 8])}

        for i in range(max(1, int(batch_size))):
            s = (seed + i) % (2**32)
            samples = KSampler.sample(unet, s, steps, cfg, sampler, scheduler, pos, neg, latent, denoise)[0]
            img = comfy_to_pil(VAEDecode.decode(vae, samples)[0])
            outputs.append(img)
            seeds.append(s)
            if on_image:
                on_image(i, img)
    return outputs, seeds


print(f"{MODEL_NAME} loaded successfully.")

In [ ]:
# Cell 3 (optional): quick test edit inside the notebook
%cd /content/ComfyUI

import os
from PIL import Image
from IPython.display import display

if "run_qwen_edit" not in globals():
    raise RuntimeError("Run cell 2 (model loader) first.")

# Upload your own image to /content/input.png, otherwise a flat placeholder is used
INPUT_PATH = "/content/input.png"
if os.path.exists(INPUT_PATH):
    input_pil = Image.open(INPUT_PATH).convert("RGB")
else:
    input_pil = Image.new("RGB", (768, 768), (120, 140, 160))

positive = "transform the scene into a dramatic studio portrait with rim lighting"
negative = ""

torch.cuda.empty_cache()
images, seeds = run_qwen_edit(positive, negative, input_pil=input_pil, width=768, height=768, **{
    k: AIO_DEFAULTS[k] for k in ("steps", "cfg", "sampler", "scheduler", "denoise")
})
print("seed:", seeds[0])
display(images[0])

In [ ]:
# Cell 4: API server for the web frontend (Cloudflare tunnel)
%cd /content/ComfyUI

import os
import io
import re
import gc
import time
import uuid
import base64
import threading
import subprocess
import traceback

import torch
import comfy.utils
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import nest_asyncio

nest_asyncio.apply()

if "run_qwen_edit" not in globals():
    raise RuntimeError("Models are not loaded in this runtime. Run cell 2 first, then this cell.")

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)


class GenerateRequest(BaseModel):
    positive: str = ""
    negative: str = ""
    steps: int = AIO_DEFAULTS["steps"]
    cfg: float = AIO_DEFAULTS["cfg"]
    width: int = AIO_DEFAULTS["width"]
    height: int = AIO_DEFAULTS["height"]
    seed: int = -1
    sampler: str = AIO_DEFAULTS["sampler"]
    scheduler: str = AIO_DEFAULTS["scheduler"]
    denoise: float = AIO_DEFAULTS["denoise"]
    batch_size: int = 1
    checkpoint: str | None = None
    loras: list = []
    reference_image: str | None = None


def decode_base64_image(b64_str):
    if not b64_str:
        return None
    cleaned = re.sub(r"^data:image/[\w.+-]+;base64,", "", b64_str.strip())
    return Image.open(io.BytesIO(base64.b64decode(cleaned))).convert("RGB")


def pil_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")


# Generation runs in a background job. Cloudflare quick tunnels drop requests that take
# longer than ~100 s, which a T4 easily exceeds, so the frontend polls /job/{id} instead.
JOBS = {}
gpu_lock = threading.Lock()
current_job = {"id": None}


def progress_hook(value, total, *args, **kwargs):
    job = JOBS.get(current_job["id"])
    if job and total:
        job["step"], job["total_steps"] = int(value), int(total)


comfy.utils.set_progress_bar_global_hook(progress_hook)


def run_job(job_id, req):
    job = JOBS[job_id]
    with gpu_lock:
        current_job["id"] = job_id
        job["status"] = "running"
        try:
            batch = max(1, min(int(req.batch_size or 1), 8))
            steps = max(1, min(int(req.steps), 50))
            input_pil = decode_base64_image(req.reference_image)
            print(f"[{job_id}] {'edit' if input_pil else 'txt2img'}: {steps} steps, cfg={req.cfg}, "
                  f"{req.sampler}/{req.scheduler}, batch={batch}")

            gc.collect()
            torch.cuda.empty_cache()

            def on_image(i, img):
                job["images"].append(pil_to_b64(img))
                job["done"] = i + 1

            _, seeds = run_qwen_edit(
                req.positive, req.negative, input_pil=input_pil,
                width=req.width, height=req.height, steps=steps, cfg=req.cfg,
                sampler=req.sampler, scheduler=req.scheduler, seed=req.seed,
                denoise=req.denoise, batch_size=batch, on_image=on_image,
            )
            job["seeds"] = seeds
            job["status"] = "done"
            print(f"[{job_id}] done")
        except Exception as e:
            traceback.print_exc()
            job["status"] = "error"
            job["error"] = str(e)
        finally:
            current_job["id"] = None
            gc.collect()
            torch.cuda.empty_cache()


def job_payload(job_id):
    job = JOBS[job_id]
    out = {
        "job_id": job_id,
        "status": job["status"],
        "batch_size": job["batch_size"],
        "done": job["done"],
        "step": job["step"],
        "total_steps": job["total_steps"],
    }
    if job["status"] == "done":
        out["images"] = job["images"]
        out["image"] = job["images"][0] if job["images"] else None
        out["seeds"] = job["seeds"]
        out["seed"] = job["seeds"][0] if job["seeds"] else None
    if job["status"] == "error":
        out["error"] = job["error"]
    return out


@app.get("/")
def root():
    return {"status": "ok", "model": MODEL_NAME, "mode": "qwen_image_edit"}


@app.get("/user_models")
def get_user_models():
    return {
        "checkpoints": [MODEL_NAME],
        "loras": [],
        "model_type": "qwen_image_edit",
        "defaults": AIO_DEFAULTS,
        "samplers": ["sa_solver", "euler", "euler_ancestral", "res_multistep", "er_sde", "dpmpp_2m"],
        "schedulers": ["beta", "simple", "normal", "sgm_uniform", "karras"],
    }


@app.post("/generate")
def generate(req: GenerateRequest):
    # Drop finished jobs older than 30 min
    now = time.time()
    for jid in [j for j, v in JOBS.items() if v["status"] in ("done", "error") and now - v["created"] > 1800]:
        JOBS.pop(jid, None)

    job_id = uuid.uuid4().hex[:12]
    JOBS[job_id] = {
        "status": "queued", "created": now, "images": [], "seeds": [], "error": None,
        "batch_size": max(1, min(int(req.batch_size or 1), 8)), "done": 0, "step": 0, "total_steps": 0,
    }
    threading.Thread(target=run_job, args=(job_id, req), daemon=True).start()
    return job_payload(job_id)


@app.get("/job/{job_id}")
def get_job(job_id: str):
    if job_id not in JOBS:
        return {"status": "error", "error": "Unknown job (the server may have restarted)"}
    return job_payload(job_id)


def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")


# Clean up a previous run of this cell
!fuser -k 8000/tcp >/dev/null 2>&1 || true
!pkill -f cloudflared >/dev/null 2>&1 || true
time.sleep(2)

if not os.path.exists("/usr/local/bin/cloudflared"):
    print("Installing Cloudflare tunnel...")
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared

threading.Thread(target=start_server, daemon=True).start()
time.sleep(4)

cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print("Starting tunnel, waiting for address...")
url = None
start_time = time.time()
while time.time() - start_time < 60:
    line = cf_proc.stdout.readline()
    if not line:
        break
    match = re.search(r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com", line)
    if match:
        url = match.group(0)
        break

# Keep draining cloudflared output so its pipe never fills up and blocks the tunnel
threading.Thread(target=lambda: [None for _ in cf_proc.stdout], daemon=True).start()

if url:
    print("=" * 50)
    print(f"READY: {url}")
    print("Paste this URL into the frontend's Backend URL field.")
    print("=" * 50)
else:
    print("Failed to retrieve Cloudflare tunnel URL.")

try:
    while True:
        time.sleep(30)
        print(".", end="", flush=True)
except KeyboardInterrupt:
    cf_proc.terminate()
    print("\nStopped.")